# Final Project

## Part 0 Project Proposal
Before we do the project, you will need to create a repository
1. Create the project remotely on GitHub.  You will want to include a README
2. Install GitHub Desktop.
3. From the remote repo, clone down to your local and use github desktop to create it.
4. Copy the notebook into your local repo
5. Commit and push

***Invite me and the GA to your repo***

As you make changes locally, commit and push them. 

To complete the proposal. You will record the following in your README

1. The kind of data you want to find
2. The question/s you want to answer with the data.
3. URLs for the site/s that have the data you want.

***Note: You must aquire the data via an API or Web scraping.  Downloading a file will not get points.***

All code for your project will be recorded in this note book. Create extra code cells as needed.


# Part 1: Data aquisition
    1. Get raw data and put it into files. If needed, gather a representative amount of data.  Then append additional data as available.
    


In [2]:
# Part 1: Put code here
import requests
import pandas as pd
import pickle
from datetime import datetime, timezone

# User-Agent is REQUIRED by the OSRS Wiki API policy
HEADERS = {
    'User-Agent': 'DataPipelineProject - hbindel42103@gmail.com',
}

def fetch_ge_data():
    print("Fetching item metadata and latest prices...")
    
    try:
        # 1. Get Item Mapping (Metadata)
        mapping_url = "https://prices.runescape.wiki/api/v1/osrs/mapping"
        mapping_res = requests.get(mapping_url, headers=HEADERS)
        mapping_res.raise_for_status()
        mapping_data = mapping_res.json()
        
        # 2. Get Latest Prices (Real-time spreads)
        latest_url = "https://prices.runescape.wiki/api/v1/osrs/latest"
        latest_res = requests.get(latest_url, headers=HEADERS)
        latest_res.raise_for_status()
        latest_data = latest_res.json()['data']
        
        # 3. Process into DataFrames
        df_mapping = pd.DataFrame(mapping_data)
        
        price_list = []
        for item_id, prices in latest_data.items():
            prices['id'] = int(item_id)
            price_list.append(prices)
        df_prices = pd.DataFrame(price_list)
        
        # Merge Metadata with Prices
        master_df = pd.merge(df_mapping, df_prices, on='id', how='left')
        
        # Use timezone-aware UTC datetime
        master_df['collected_at'] = datetime.now(timezone.utc)
        
        return master_df

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def save_cache(df, filename="ge_data_cache.pkl"):
    if df is not None:
        with open(filename, 'wb') as f:
            pickle.dump(df, f)
        print(f"Data cached successfully to {filename}")

if __name__ == "__main__":
    data = fetch_ge_data()
    save_cache(data)
    if data is not None:
        print("\nInitial Data Sample:")
        print(data[['name', 'high', 'low', 'members']].head())

Fetching item metadata and latest prices...
Data cached successfully to ge_data_cache.pkl

Initial Data Sample:
                    name          high           low  members
0         3rd age amulet  5.550000e+07  5.388723e+07     True
1            3rd age axe  2.147484e+09  2.147484e+09     True
2            3rd age bow  1.498000e+09  1.401169e+09     True
3          3rd age cloak  3.888880e+08  3.757770e+08     True
4  3rd age druidic cloak  9.788880e+08  9.605330e+08     True


## Part 2 Data processing

1. Process the raw data and store results in a file as needed.
2. Do analysis of the data
   

In [4]:
#Part 2: Put code here
# Part 2: Data Processing and Analysis
import pandas as pd
import pickle

# 1. Load the raw data
with open("ge_data_cache.pkl", 'rb') as f:
    df = pickle.load(f)

# 2. Basic Cleaning: Remove rows missing prices or having "junk" values (under 100gp)
# This prevents extreme % margin outliers (e.g., 2gp items showing 5,000,000% profit)
df = df.dropna(subset=['high', 'low'])
df = df[df['low'] > 100] 

# 3. Feature Engineering: Calculate Spread and Normalized Margin
df['price_spread'] = df['high'] - df['low']
df['margin_pct'] = (df['price_spread'] / df['low']) * 100

# 4. Filter for viable flips: Positive spread and reasonable margins
# We sort by margin_pct to see the best volatility opportunities
flipping_candidates = df[df['price_spread'] > 0].sort_values(by='margin_pct', ascending=False)

# 5. Visual Formatting: Display the top results clearly
pd.set_option('display.expand_frame_repr', False) # Keeps columns in one row
analysis_df = flipping_candidates[['id', 'name', 'high', 'low', 'price_spread', 'margin_pct', 'members']]

print("Data Processing Complete. Realistic Flipping Margins:")
print(analysis_df.head(10))

# 6. Save processed data for the Visualization stage
analysis_df.to_pickle("processed_ge_data.pkl")



         id                      name      high     low  price_spread    margin_pct  members
245   26353            Ancient mix(1)  700000.0  1000.0      699000.0  69900.000000     True
3926   8506  Teak armchair (flatpack)   50000.0   101.0       49899.0  49404.950495     True
336    6891                Arena book  746987.0  2500.0      744487.0  29779.480000     True
3831  21997     Super antifire mix(1)  219999.0   800.0      219199.0  27399.875000     True
1034   9642          Citizen trousers  200000.0   757.0      199243.0  26320.079260     True
910    3333         Bruise blue snelm  950000.0  4000.0      946000.0  23650.000000     True
3179   8500  Rocking chair (flatpack)   50000.0   216.0       49784.0  23048.148148     True
1133  20243                Crier bell  999000.0  4500.0      994500.0  22100.000000     True
550    1233           Black dagger(p)   38696.0   175.0       38521.0  22012.000000     True
1473   9729          Elemental helmet  300000.0  1500.0      298500.0 

Data Manipulation Process
What manipulations are being performed?
I have implemented four key transformations to turn raw API data into a usable dataset:

Outlier Filtering: I removed all items with a price below 100gp. This is a critical step because items with a 1gp or 2gp "low" price create massive, mathematically impossible profit percentages that ruin the analysis.

Null Value Cleanup: I used dropna to ensure every item in the final set has both a "high" (buy) and "low" (sell) price, preventing calculation errors.

Feature Engineering: I derived two new metrics:

price_spread: The absolute gold profit per item.

margin_pct: The percentage of profit relative to the investment cost.

Display Optimization: I used pd.set_option to prevent Pandas from wrapping text, allowing for a clean, single-row view of the data.

Why manipulate the data this way?
Raw data from the Grand Exchange is "noisy." An item might show a huge spread, but if it's a "junk" item with no liquidity, it isn't a real flipping opportunity. By filtering for prices > 100gp, I ensure the results represent actual tradeable goods. Calculating the Margin Percentage is the most important step because it allows us to compare the volatility of a cheap item (like a Rune Scimitar) against an expensive item (like a Godwars piece) on a fair, scaleable basis.

How this makes the data usable for the final presentation
These manipulation steps directly prepare the data for my three research questions:

Market Volatility: By sorting by margin_pct, I can now generate a Bar Chart showing the top 10 most volatile items in the game.

Economic Correlation: The cleaned data allows me to isolate specific Item IDs (like Runite Ore and Rune Bars) to perform a Pearson Correlation test without junk data skewing the results.

Optimal Timing: The refined dataset is now light enough to be merged with time-series data to identify when these high-margin opportunities are most active during the day.

## Part 3 Visualization

1. Create good graphs
2. Give narative conclusions explaining what is being demonstrated in the visualizations.

   

In [3]:
#Part 3: Put code here

